## RGB Model Test on LWIR

The objective of this script is to test a YOLO model trained on RGB images on new sets of LWIR test images

First, let's import the necessary libraries.

In [2]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.4 MB/s eta 0:00:00


In [3]:
import os
import yaml
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### 1. Testing Preparation

We first mount our drive to point to the folder where the testing images are located.

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


Next, we load the trained model

<div class="alert alert-block alert-info">
    
<b>Note:</b> YOLOv8 automatically saves the model on training. The saved model can be found in this path where the training script is located. *runs/detect/train/exp*/weights/*

The model is automatically named as *"best.pt"*

</div>

In [8]:
model_path = os.path.join("/content/drive/MyDrive/Drone_images/RGB_Training/runs/detect/train/weights", "best.pt")
rgb_model = YOLO(model_path)

Next we define the path of the test images including the labels (Pascal VOC .xml labels and .txt class files)

In [9]:
test_images = "/content/drive/MyDrive/Drone_images/LWIR_Training/images/test"   # folder with test images
test_labels = "/content/drive/MyDrive/Drone_images/LWIR_Training/labels/test"   # folder with YOLO .txt labels
xml_labels  = "/content/drive/MyDrive/Drone_images/LWIR_Training/labels/test"   # folder with XML files

### 2. Testing the LWIR Images

We make predictions using the trained model

In [10]:
pred_results = rgb_model.predict(source = test_images, imgsz = 640, save = True)


image 1/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_100_50.jpg: 512x640 2 at_plastics, 7.1ms
image 2/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_100_56.jpg: 512x640 7 at_plastics, 5.9ms
image 3/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_100_8.jpg: 512x640 2 at_plastics, 5.9ms
image 4/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_20.jpg: 512x640 1 ap_metal, 1 ap_plastic, 4 at_plastics, 6.4ms
image 5/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_3.jpg: 512x640 2 ap_metals, 1 ap_plastic, 3 at_plastics, 6.0ms
image 6/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_50.jpg: 512x640 1 ap_metal, 2 ap_plastics, 9 at_plastics, 8.1ms
image 7/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_51.jpg: 512x640 1 ap_metal, 1 ap_plastic, 9 at_plastics, 6.

### 3. Model Evaluation

First, we create aand save structured configuration YAML file that will help us in evaluating the model.

In [11]:
class_names = ['ap_metal', 'ap_plastic', 'at_metal', 'at_plastic']

In [12]:
data = {
    "path":"/content/drive/MyDrive/Drone_images/LWIR_Training",
    "train": os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/train"),
    "val": os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/val"),
    "test": os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/test"),
    "names": class_names,
}

yaml_path = os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Testing", "eval.yaml")
with open(yaml_path, "w") as f:

    yaml.dump(data, f, default_flow_style = False)

We finally evaluate the performance of our model by printing and saving the precision, recall, 50% and 90% mean Average Precision (mAP) scores.

In [13]:
results = rgb_model.val(data = yaml_path, split = "test",
    imgsz = 640, batch = 16, save_json = True, plots = True)

Ultralytics 8.3.202 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.8±1.0 ms, read: 76.1±35.9 MB/s, size: 133.1 KB)
val: Scanning /content/drive/MyDrive/Drone_images/LWIR_Training/labels/test.cache... 561 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 561/561 720.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.3it/s 10.8s
                   all        561       5143      0.727       0.43      0.468      0.264
              ap_metal        336        539      0.657      0.234      0.253      0.123
            ap_plastic        353        786        0.6      0.248      0.272      0.128
              at_metal        389        588      0.809      0.568      0.613      0.372
            at_plastic        548       3230      0.842      0.671      0.734      0.432
Speed: 0.5ms preprocess, 3.8ms inference, 0.0ms loss, 2.1ms postprocess per image
Saving /content

We save the model evaluation results for future use.

In [14]:
!cp -r /content/runs/detect/val /content/drive/MyDrive/Drone_images/LWIR_Testing/rgb_model_predictions_on_lwir/

And then extract the individual metrics before converting to dataframe and eventually saving the results as a csv file.

In [15]:
precision = results.box.p
recall = results.box.r
ap50 = results.box.ap50
ap5095 = results.box.ap
class_names = results.names

In [16]:
df = pd.DataFrame({
    "class": [class_names[i] for i in range(len(precision))],
    "precision": precision,
    "recall": recall,
    "map50": ap50,
    "map50_95": ap5095,
    "model": ["RGB on LWIR Images"] * len(precision)
    })

RESULTS = "/content/drive/MyDrive/Drone_images/LWIR_Testing"
output_file = os.path.join(RESULTS, "rgb_model_on_lwir_images_metrics.csv")
df.to_csv(output_file, index=False)